In [1]:
import torchxrayvision as xrv
import torch, torchvision
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt

# Projection algorithms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
import umap.umap_ as umap
import pacmap

In [2]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.widgets import RectangleSelector

In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import numpy as np
import pandas as pd

### Constants and Configs

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed seed for reproducibility
RANDOM_STATE = 42

In [5]:
PATHOLOGIES = [
    "Lung Opacity",
    "Fracture",
    # TODO: check if we should add this or not
    #"Nodule/Mass",
    "Pneumothorax"
]

### Load and Prepare

In [6]:
def load_dataset(image_path: str = "./nih_data_small", datasetClass = xrv.datasets.NIH_Google_Dataset):
    transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
    d_nih = datasetClass(imgpath=image_path, transform=transform, views=["PA", "AP"])
    print(f'found {len(d_nih)} images')
    return d_nih

In [7]:
NIH_GOOGLE_DATASET = load_dataset()
DATASET_PATHOLOGIES = list(NIH_GOOGLE_DATASET.pathologies)
DATASET_LABEL_INDICES = {
        name: DATASET_PATHOLOGIES.index(name)
        for name in PATHOLOGIES
}
DATASET_INDICES_LABEL = {DATASET_LABEL_INDICES[l]:l for l in DATASET_LABEL_INDICES}

found 1695 images


In [8]:
DATASET_INDICES_LABEL

{0: 'Lung Opacity', 1: 'Fracture', 3: 'Pneumothorax'}

In [9]:
def create_image_feature_space_and_label(dataset, dataset_indices_label = DATASET_INDICES_LABEL):
    image_features_list = []
    labels_list = []
    for i in range(len(dataset)):
        sample = dataset[i]
        label = np.asarray(sample["lab"]).reshape(-1)

        img = np.asarray(sample["img"]).squeeze()
        image_features_list.append(
                        img.astype(np.float32).reshape(-1)
            )
            
        labels_list.append(
                    label
                )

    image_features = np.stack(
        image_features_list,
        axis=0
    )
    labels = np.array(labels_list)
    return image_features, labels


In [10]:
NIH_GOOGLE_IMAGE_FEATURES, NIH_GOOGLE_LABELS_LIST = create_image_feature_space_and_label(NIH_GOOGLE_DATASET)

### Load model

In [11]:
def load_model(model_weights: str = "densenet121-res224-all"):
    model = xrv.models.DenseNet(weights=model_weights)
    model.eval()
    return model.to(DEVICE)

In [12]:
DENSENET121_RES244_CHEX_MODEL = load_model()
DENSENET121_RES244_CHEX_MODEL.eval()

XRV-DenseNet121-densenet121-res224-all

In [13]:
def get_model_to_dataset_indices(model = DENSENET121_RES244_CHEX_MODEL, dataset_label_indices=DATASET_LABEL_INDICES):
    return {
        model_idx: dataset_label_indices[pathology]
        for model_idx, pathology in enumerate(model.pathologies)
        if pathology in dataset_label_indices
    }   

In [14]:
MODEL_TO_DATASET_INDICES = get_model_to_dataset_indices()
DATASET_TO_MODEL_INDICES = {MODEL_TO_DATASET_INDICES[key]:key for key in MODEL_TO_DATASET_INDICES}
print(MODEL_TO_DATASET_INDICES)
print(DATASET_TO_MODEL_INDICES)

{3: 3, 15: 1, 16: 0}
{3: 3, 1: 15, 0: 16}


### Prediction of the model

In [15]:
DATASET_LABEL_INDICES

{'Lung Opacity': 0, 'Fracture': 1, 'Pneumothorax': 3}

In [16]:
def calculate_model_predictions_score(
    feature_space,
    model,
    batch_size=32
):
    model.eval()

    # Convert flattened images to (N, 1, 224, 224)
    images = torch.as_tensor(feature_space, dtype=torch.float32)
    images = images.reshape(-1, 1, 224, 224)

    # Run inference in batches
    all_outputs = []

    with torch.no_grad():
        for start_idx in range(0, len(images), batch_size):
            end_idx = start_idx + batch_size

            batch = images[start_idx:end_idx].to(DEVICE)

            batch_outputs = model(batch)

            # Move each batch back to CPU immediately
            batch_outputs = batch_outputs.cpu().numpy()


            all_outputs.append(batch_outputs)

    # Combine all batches
    selected_outputs = np.concatenate(all_outputs, axis=0)
    return selected_outputs

In [17]:
MODEL_SCORE_PREDICTIONS = calculate_model_predictions_score(NIH_GOOGLE_IMAGE_FEATURES,
                                                       DENSENET121_RES244_CHEX_MODEL)

### Downprojection

In [18]:
class Downprojector:

    def __init__(self, image_features):
        self.projection_cache = {}
        self.image_features = image_features


    def get_feature_matrix(self, feature_space):
        return self.image_features

    """
    PCA is used as a preprocessing step for the non-linear
        projection techniques.
    """
    def prepare_pca_features(self, feature_space):
        cache_key = f"{feature_space}_pca"
         
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]
         
        X = self.get_feature_matrix(feature_space)
         
             # Keep at most 50 dimensions before expensive
             # non-linear dimensionality reduction.
        n_components = min(
                 50,
                 X.shape[0],
                 X.shape[1]
             )
         
        pca = PCA(
                 n_components=n_components,
                 random_state=RANDOM_STATE
        )
         
        X_pca = pca.fit_transform(X)
         
        self.projection_cache[cache_key] = {
                 "data": X_pca,
                 "pca": pca
             }
         
        return self.projection_cache[cache_key]

    def calculate_projection(self, feature_space: str, method: str):
        cache_key = (
                f"{feature_space}_{method}"
            )
        
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]


        if method == "PCA":

            X = self.get_feature_matrix(feature_space)

            reducer = PCA(
                n_components=2,
                random_state=RANDOM_STATE
            )

            embedding = reducer.fit_transform(X)
        elif method == "t-SNE":
        
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]
        
                # t-SNE requires perplexity < number of samples.
            perplexity = min(
                    30,
                    max(5, (X_pca.shape[0] - 1) // 3)
                )
        
            reducer = TSNE(
                    n_components=2,
                    perplexity=perplexity,
                    init="pca",
                    learning_rate="auto",
                    random_state=RANDOM_STATE
                )
        
            embedding = reducer.fit_transform(
                    X_pca
                )
        elif method == "UMAP":
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]

            n_neighbors = min(
                    15,
                    max(2, X_pca.shape[0] - 1)
                )

            reducer = umap.UMAP(
                    n_components=2,
                    n_neighbors=n_neighbors,
                    min_dist=0.1,
                    metric="euclidean",
                    random_state=RANDOM_STATE
                )

            embedding = reducer.fit_transform(
                    X_pca
                )
        else:
            raise ValueError(
                f"Unknown projection method: {method}"
                )
        self.projection_cache[cache_key] = embedding
        return embedding

### Run first visualization

In [50]:
def visualize_space(feature_space, projection_method,
                    label_name="Pneumothorax",
                     labels_list=NIH_GOOGLE_LABELS_LIST,
                     dataset_label_indices = DATASET_LABEL_INDICES):
    label_index = dataset_label_indices[label_name]
    label_pathologie_label_output = labels_list[:, label_index]
    idx_pathologie = label_pathologie_label_output == 1
    idx_non_pathologie = label_pathologie_label_output == 0

    fig, ax = plt.subplots(figsize=(11, 8))

    ax.scatter(feature_space[idx_pathologie,0], feature_space[idx_pathologie,1], label=label_name, c='red')
    ax.scatter(feature_space[idx_non_pathologie,0], feature_space[idx_non_pathologie,1], label='Not effected', c='blue',
                alpha=0.1)

    ax.set_title(f"{projection_method} Projection Colored")
    ax.set_xlabel(f"{projection_method} 1")
    ax.set_ylabel(f"{projection_method} 2")
    ax.legend()
    plt.show()

In [20]:
downprojector = Downprojector(NIH_GOOGLE_IMAGE_FEATURES)

In [55]:
def create_viewer(downprojector: Downprojector, visualization_fn):

    plot_output = widgets.Output()

    plot_button = widgets.Button(
        description="Create selection plot",
        button_style="primary",
        icon="refresh"
    )

    pathology_dropdown = widgets.Dropdown(
        options=PATHOLOGIES,
        value=PATHOLOGIES[0],
        description="Pathology:",
        layout=widgets.Layout(width="350px")
    )

    projection_dropdown = widgets.Dropdown(
        options=[
            "PCA",
            "t-SNE",
            "UMAP"
        ],
        value="PCA",
        description="Projection:",
        layout=widgets.Layout(width="300px")
    )

    def create_selection_plot(_=None):

        with plot_output:
            clear_output(wait=True)

            pathology = pathology_dropdown.value
            method = projection_dropdown.value

            # Calculate the requested projection if it has not already
            # been calculated.
            embedding = downprojector.calculate_projection(
                "feature",
                method
            )
            visualization_fn(embedding, method, pathology)
            

    plot_button.on_click(create_selection_plot)

    display(
    widgets.VBox([

        widgets.HBox([
            projection_dropdown
        ]),

        widgets.HBox([
            pathology_dropdown
        ]),



        plot_button,

        plot_output,
    ])
)

In [53]:
create_viewer(downprojector, visualize_space)

### Visualize predictions


In [56]:
def visualize_space_prediction_scores(feature_space, projection_method,label_name="Pneumothorax", 
                                      predictions=MODEL_SCORE_PREDICTIONS,
                    dataset_label_indices = DATASET_LABEL_INDICES,
                    dataset_to_model_indices=DATASET_TO_MODEL_INDICES):

    # Find the prediction column for this finding
    dataset_label_index = dataset_label_indices[label_name]
   
    model_label_index = dataset_to_model_indices[dataset_label_index]

    # Get the model score for each image
    scores = predictions[:, model_label_index]

    plt.figure(figsize=(10, 7))

    scatter = plt.scatter(
        feature_space[:, 0],
        feature_space[:, 1],
        c=scores,
        cmap="viridis",
        alpha=0.8,
        s=20
    )

    plt.colorbar(scatter, label=f"Model score: {label_name}")
    plt.title(f"{projection_method} Projection Colored by {label_name} Score")
    plt.xlabel(f"{projection_method} 1")
    plt.ylabel(f"{projection_method} 2")
    plt.show()

In [57]:
create_viewer(downprojector, visualize_space_prediction_scores)

### Compare Prediction with Dataset

In [38]:
def convert_to_dataset_label_list(model_prediction_score, dataset_labels_list = NIH_GOOGLE_LABELS_LIST,
                                  dataset_to_model_indices=DATASET_TO_MODEL_INDICES
                                  ):
    new_prection_list = np.zeros_like(dataset_labels_list)
    for selectedDatasetIndex in dataset_to_model_indices:
        selectedModelIndex = dataset_to_model_indices[selectedDatasetIndex]
        new_prection_list[:, selectedDatasetIndex] = model_prediction_score[:, selectedModelIndex]
    return new_prection_list

MODEL_SCORE_PREDICTIONS_CONVERTED = convert_to_dataset_label_list(MODEL_SCORE_PREDICTIONS)


In [39]:
def calulcate_metrics_for_model(
    model_prediction,
    dataset_labels_list=NIH_GOOGLE_LABELS_LIST,
    threshold=0.5,
    dataset_label_to_indicies = DATASET_LABEL_INDICES
):

    # Convert scores into binary predictions
    binary_prediction = (model_prediction >= threshold).astype(int)


    # Store metrics for every label
    metrics = []

    for label in PATHOLOGIES:
        label_idx = dataset_label_to_indicies[label]
        # Ground truth
        y_true = dataset_labels_list[:, label_idx]

        # Prediction
        y_pred = binary_prediction[:, label_idx]

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        # Metrics
        precision = precision_score(
            y_true, y_pred, zero_division=0
        )

        recall = recall_score(
            y_true, y_pred, zero_division=0
        )

        f1 = f1_score(
            y_true, y_pred, zero_division=0
        )

        accuracy = accuracy_score(
            y_true, y_pred
        )

        # Specificity = TN / (TN + FP)
        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        )

        # False-positive rate
        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else 0
        )

        # False-negative rate
        fnr = (
            fn / (fn + tp)
            if (fn + tp) > 0
            else 0
        )

        # ROC-AUC can fail if only one class exists
        try:
            auc = roc_auc_score(
                y_true,
                model_prediction[:, label_idx]
            )
        except ValueError:
            auc = np.nan


        metrics.append({
            "label": label,
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "FPR": fpr,
            "FNR": fnr,
            "F1": f1,
            "ROC_AUC": auc
        })

    metrics_df = pd.DataFrame(metrics)

    # Overall metrics
    overall_metrics = {
        "macro_accuracy": metrics_df["accuracy"].mean(),
        "macro_precision": metrics_df["precision"].mean(),
        "macro_recall": metrics_df["recall"].mean(),
        "macro_specificity": metrics_df["specificity"].mean(),
        "macro_F1": metrics_df["F1"].mean(),
        "macro_ROC_AUC": metrics_df["ROC_AUC"].mean(),

        "micro_precision": precision_score(
            dataset_labels_list.flatten(),
            binary_prediction.flatten(),
            zero_division=0
        ),

        "micro_recall": recall_score(
            dataset_labels_list.flatten(),
            binary_prediction.flatten(),
            zero_division=0
        ),

        "micro_F1": f1_score(
            dataset_labels_list.flatten(),
            binary_prediction.flatten(),
            zero_division=0
        )
    }

    return metrics_df, overall_metrics

In [40]:
calulcate_metrics_for_model(MODEL_SCORE_PREDICTIONS_CONVERTED, NIH_GOOGLE_LABELS_LIST)

(          label   TP    TN   FP  FN  accuracy  precision    recall  \
 0  Lung Opacity  569   807  287  32  0.811799   0.664720  0.946755   
 1      Fracture   31  1092  543  29  0.662537   0.054007  0.516667   
 2  Pneumothorax   66   952  671   6  0.600590   0.089552  0.916667   
 
    specificity       FPR       FNR        F1   ROC_AUC  
 0     0.737660  0.262340  0.053245  0.781057  0.931824  
 1     0.667890  0.332110  0.483333  0.097792  0.647085  
 2     0.586568  0.413432  0.083333  0.163164  0.841377  ,
 {'macro_accuracy': np.float64(0.6916420845624386),
  'macro_precision': np.float64(0.2694262778717698),
  'macro_recall': np.float64(0.7933629136624144),
  'macro_specificity': np.float64(0.6640393184964167),
  'macro_F1': np.float64(0.34733772165698173),
  'macro_ROC_AUC': np.float64(0.8067621674167821),
  'micro_precision': 0.30733733271804337,
  'micro_recall': 0.6901554404145078,
  'micro_F1': 0.42528735632183906})